# Predict tournament performance

Simulate the 2026 World Cup from the calculated matchup odds and count how often each team reaches each phase.

Output: `3.SimulationStudy-Python/PredictPerformance.json`.

In [ ]:
from pathlib import Path
from itertools import combinations
import json
import random

import pandas as pd

In [ ]:
def find_project_root(start):
    start = Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "3.SimulationStudy-Python").exists() and (path / "4.Dashboard").exists():
            return path
    raise FileNotFoundError("Could not find WC2026Forecast project root")


PROJECT_ROOT = find_project_root(Path.cwd())
SIM_DIR = PROJECT_ROOT / "3.SimulationStudy-Python"

ODDS_CANDIDATES = [
    PROJECT_ROOT / "4.Dashboard" / "wc2026_predictions.json",
    PROJECT_ROOT / "4.Dashboard" / "data" / "wc2026_predictions.json",
    PROJECT_ROOT / "1.DataCleaning-R" / "Data" / "JSON" / "WC2026Predictions.json",
]
ODDS_PATH = next((path for path in ODDS_CANDIDATES if path.exists()), None)
if ODDS_PATH is None:
    raise FileNotFoundError("Could not find wc2026_predictions.json")

OUTPUT_PATH = SIM_DIR / "PredictPerformance.json"

N_ITERATIONS = 10000
SEED = 2026
KNOCKOUT_MAX_ADVANCE_PROB = 0.80

PROJECT_ROOT, ODDS_PATH


## Tournament setup

Groups are copied from the dashboard and normalized to the team names used in the prediction JSON. The knockout layout follows the 2026 format: top two from each group plus the eight best third-place teams advance to a Round of 32.

In [ ]:
NAME_MAP = {
    "Korea Republic": "South Korea",
    "Turkiye": "Turkey",
    "Cote d'Ivoire": "Ivory Coast",
    "IR Iran": "Iran",
}

GROUPS_RAW = {
    "A": ["Mexico", "South Africa", "Korea Republic", "Czechia"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkiye"],
    "E": ["Germany", "Curacao", "Cote d'Ivoire", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "IR Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

GROUPS = {
    group: [NAME_MAP.get(team, team) for team in teams]
    for group, teams in GROUPS_RAW.items()
}

RATINGS_RAW = {
    "Argentina": 95, "France": 94, "Spain": 92, "Brazil": 91, "England": 90, "Portugal": 89,
    "Germany": 88, "Netherlands": 87, "Belgium": 85, "Uruguay": 84, "Croatia": 83, "Colombia": 82,
    "Switzerland": 80, "Morocco": 79, "Japan": 78, "United States": 77, "Mexico": 76, "Senegal": 75,
    "Austria": 74, "Scotland": 72, "Sweden": 71, "Ecuador": 70, "Cote d'Ivoire": 69, "Korea Republic": 68,
    "Australia": 67, "Turkiye": 66, "Paraguay": 65, "Canada": 64, "Czechia": 63, "Tunisia": 62, "Egypt": 61,
    "Algeria": 60, "Saudi Arabia": 59, "Qatar": 58, "Bosnia and Herzegovina": 57, "Ghana": 56,
    "IR Iran": 55, "South Africa": 54, "Panama": 53, "Norway": 52, "Uzbekistan": 51, "Jordan": 50,
    "Iraq": 49, "Congo DR": 48, "Cabo Verde": 47, "New Zealand": 46, "Curacao": 45, "Haiti": 44,
}
RATINGS = {NAME_MAP.get(team, team): rating for team, rating in RATINGS_RAW.items()}

GROUP_MATCH_ORDER = [(0, 1), (2, 3), (3, 1), (0, 2), (1, 2), (3, 0)]

R32_TEMPLATE = [
    {"match": 73, "a": "2A", "b": "2B", "round": "R32"},
    {"match": 74, "a": "1E", "b": "3ABCDF", "round": "R32"},
    {"match": 75, "a": "1F", "b": "2C", "round": "R32"},
    {"match": 76, "a": "1C", "b": "2F", "round": "R32"},
    {"match": 77, "a": "1I", "b": "3CDFGH", "round": "R32"},
    {"match": 78, "a": "2E", "b": "2I", "round": "R32"},
    {"match": 79, "a": "1A", "b": "3CEFHI", "round": "R32"},
    {"match": 80, "a": "1L", "b": "3EHIJK", "round": "R32"},
    {"match": 81, "a": "1D", "b": "3BEFIJ", "round": "R32"},
    {"match": 82, "a": "1G", "b": "3AEHIJ", "round": "R32"},
    {"match": 83, "a": "2K", "b": "2L", "round": "R32"},
    {"match": 84, "a": "1H", "b": "2J", "round": "R32"},
    {"match": 85, "a": "1B", "b": "3EFGIJ", "round": "R32"},
    {"match": 86, "a": "1J", "b": "2H", "round": "R32"},
    {"match": 87, "a": "1K", "b": "3DEIJL", "round": "R32"},
    {"match": 88, "a": "2D", "b": "2G", "round": "R32"},
]

NEXT_ROUNDS = {
    "R16": [(90, 73, 75), (89, 74, 77), (91, 76, 78), (92, 79, 80), (93, 83, 84), (94, 81, 82), (95, 86, 88), (96, 85, 87)],
    "QF": [(97, 89, 90), (98, 93, 94), (99, 91, 92), (100, 95, 96)],
    "SF": [(101, 97, 98), (102, 99, 100)],
    "F": [(104, 101, 102)],
}

THIRD_PLACE_ASSIGNMENT_TABLE = {
    'ABCDEFGH': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3G', 87: '3D'},
    'ABCDEFGI': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCDEFGJ': {74: '3D', 77: '3F', 79: '3C', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCDEFGK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCDEFGL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDEFHI': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3E', 87: '3D'},
    'ABCDEFHJ': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3D'},
    'ABCDEFHK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3D'},
    'ABCDEFHL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3F', 87: '3L'},
    'ABCDEFIJ': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCDEFIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3I'},
    'ABCDEFIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABCDEFJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCDEFJL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDEFKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABCDEGHI': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCDEGHJ': {74: '3C', 77: '3D', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCDEGHK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCDEGHL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDEGIJ': {74: '3C', 77: '3D', 79: '3E', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCDEGIK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCDEGIL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDEGJK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABCDEGJL': {74: '3C', 77: '3D', 79: '3E', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDEGKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDEHIJ': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCDEHIK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3I'},
    'ABCDEHIL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABCDEHJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCDEHJL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDEHKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABCDEIJK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCDEIJL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDEIKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABCDEJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDFGHI': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3D'},
    'ABCDFGHJ': {74: '3C', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3D'},
    'ABCDFGHK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3D'},
    'ABCDFGHL': {74: '3D', 77: '3F', 79: '3C', 80: '3H', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDFGIJ': {74: '3D', 77: '3F', 79: '3C', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCDFGIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCDFGIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDFGJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABCDFGJL': {74: '3D', 77: '3F', 79: '3C', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDFGKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDFHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3D'},
    'ABCDFHIK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3F', 87: '3I'},
    'ABCDFHIL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3F', 87: '3L'},
    'ABCDFHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3D'},
    'ABCDFHJL': {74: '3D', 77: '3F', 79: '3C', 80: '3H', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDFHKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3F', 87: '3L'},
    'ABCDFIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCDFIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDFIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABCDFJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDGHIJ': {74: '3C', 77: '3D', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCDGHIK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCDGHIL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDGHJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABCDGHJL': {74: '3C', 77: '3D', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDGHKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDGIJK': {74: '3D', 77: '3G', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCDGIJL': {74: '3D', 77: '3G', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDGIKL': {74: '3C', 77: '3D', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCDGJKL': {74: '3D', 77: '3G', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDHIJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCDHIJL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDHIKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABCDHJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCDIJKL': {74: '3C', 77: '3D', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEFGHI': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCEFGHJ': {74: '3C', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCEFGHK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABCEFGHL': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCEFGIJ': {74: '3C', 77: '3F', 79: '3E', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCEFGIK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCEFGIL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCEFGJK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABCEFGJL': {74: '3C', 77: '3F', 79: '3E', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCEFGKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCEFHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCEFHIK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3I'},
    'ABCEFHIL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABCEFHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCEFHJL': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEFHKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABCEFIJK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCEFIJL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEFIKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABCEFJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEGHIJ': {74: '3C', 77: '3G', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCEGHIK': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCEGHIL': {74: '3C', 77: '3H', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCEGHJK': {74: '3C', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABCEGHJL': {74: '3C', 77: '3G', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEGHKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCEGIJK': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCEGIJL': {74: '3C', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEGIKL': {74: '3A', 77: '3C', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'ABCEGJKL': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEHIJK': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCEHIJL': {74: '3C', 77: '3H', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEHIKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABCEHJKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCEIJKL': {74: '3A', 77: '3C', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ABCFGHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCFGHIK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABCFGHIL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCFGHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABCFGHJL': {74: '3C', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCFGHKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCFGIJK': {74: '3F', 77: '3G', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCFGIJL': {74: '3F', 77: '3G', 79: '3C', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCFGIKL': {74: '3C', 77: '3F', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCFGJKL': {74: '3F', 77: '3G', 79: '3C', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCFHIJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCFHIJL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCFHIKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABCFHJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCFIJKL': {74: '3C', 77: '3F', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCGHIJK': {74: '3C', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABCGHIJL': {74: '3C', 77: '3G', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCGHIKL': {74: '3C', 77: '3H', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABCGHJKL': {74: '3C', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCGIJKL': {74: '3C', 77: '3G', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABCHIJKL': {74: '3C', 77: '3H', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEFGHI': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABDEFGHJ': {74: '3D', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABDEFGHK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3E'},
    'ABDEFGHL': {74: '3D', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDEFGIJ': {74: '3D', 77: '3F', 79: '3E', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABDEFGIK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABDEFGIL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDEFGJK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABDEFGJL': {74: '3D', 77: '3F', 79: '3E', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDEFGKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDEFHIJ': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABDEFHIK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3I'},
    'ABDEFHIL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABDEFHJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABDEFHJL': {74: '3D', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEFHKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3E', 87: '3L'},
    'ABDEFIJK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABDEFIJL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEFIKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABDEFJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEGHIJ': {74: '3D', 77: '3G', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABDEGHIK': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABDEGHIL': {74: '3D', 77: '3H', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDEGHJK': {74: '3D', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABDEGHJL': {74: '3D', 77: '3G', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEGHKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDEGIJK': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABDEGIJL': {74: '3D', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEGIKL': {74: '3A', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'ABDEGJKL': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEHIJK': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABDEHIJL': {74: '3D', 77: '3H', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEHIKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABDEHJKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDEIJKL': {74: '3A', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ABDFGHIJ': {74: '3D', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABDFGHIK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABDFGHIL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDFGHJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3J'},
    'ABDFGHJL': {74: '3D', 77: '3F', 79: '3H', 80: '3J', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDFGHKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDFGIJK': {74: '3D', 77: '3G', 79: '3F', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABDFGIJL': {74: '3D', 77: '3G', 79: '3F', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDFGIKL': {74: '3D', 77: '3F', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDFGJKL': {74: '3D', 77: '3G', 79: '3F', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDFHIJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABDFHIJL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDFHIKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABDFHJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDFIJKL': {74: '3D', 77: '3F', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDGHIJK': {74: '3D', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABDGHIJL': {74: '3D', 77: '3G', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDGHIKL': {74: '3D', 77: '3H', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABDGHJKL': {74: '3D', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDGIJKL': {74: '3D', 77: '3G', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABDHIJKL': {74: '3D', 77: '3H', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABEFGHIJ': {74: '3F', 77: '3G', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABEFGHIK': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3I'},
    'ABEFGHIL': {74: '3F', 77: '3H', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABEFGHJK': {74: '3F', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3E'},
    'ABEFGHJL': {74: '3F', 77: '3G', 79: '3H', 80: '3E', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABEFGHKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3G', 87: '3L'},
    'ABEFGIJK': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABEFGIJL': {74: '3F', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABEFGIKL': {74: '3A', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'ABEFGJKL': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABEFHIJK': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABEFHIJL': {74: '3F', 77: '3H', 79: '3E', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABEFHIKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3I', 87: '3L'},
    'ABEFHJKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABEFIJKL': {74: '3A', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ABEGHIJK': {74: '3A', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'ABEGHIJL': {74: '3A', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'ABEGHIKL': {74: '3A', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'ABEGHJKL': {74: '3A', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'ABEGIJKL': {74: '3A', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ABEHIJKL': {74: '3A', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ABFGHIJK': {74: '3F', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3I'},
    'ABFGHIJL': {74: '3F', 77: '3G', 79: '3H', 80: '3I', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABFGHIKL': {74: '3A', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'ABFGHJKL': {74: '3F', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABFGIJKL': {74: '3F', 77: '3G', 79: '3I', 80: '3K', 81: '3B', 82: '3A', 85: '3J', 87: '3L'},
    'ABFHIJKL': {74: '3A', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ABGHIJKL': {74: '3A', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'ACDEFGHI': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3G', 87: '3D'},
    'ACDEFGHJ': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3J', 82: '3A', 85: '3G', 87: '3D'},
    'ACDEFGHK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3D'},
    'ACDEFGHL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3F', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEFGIJ': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ACDEFGIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3I'},
    'ACDEFGIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEFGJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ACDEFGJL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEFGKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEFHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3J', 87: '3D'},
    'ACDEFHIK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3F', 82: '3A', 85: '3E', 87: '3I'},
    'ACDEFHIL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3F', 82: '3A', 85: '3E', 87: '3L'},
    'ACDEFHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3D'},
    'ACDEFHJL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3F', 82: '3A', 85: '3J', 87: '3L'},
    'ACDEFHKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3F', 82: '3A', 85: '3E', 87: '3L'},
    'ACDEFIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3I'},
    'ACDEFIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ACDEFIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3A', 85: '3E', 87: '3L'},
    'ACDEFJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ACDEGHIJ': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ACDEGHIK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3I'},
    'ACDEGHIL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEGHJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ACDEGHJL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEGHKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEGIJK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ACDEGIJL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEGIKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEGJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDEHIJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3I'},
    'ACDEHIJL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ACDEHIKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3E', 87: '3L'},
    'ACDEHJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ACDEIJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACDFGHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3D'},
    'ACDFGHIK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3F', 82: '3A', 85: '3G', 87: '3I'},
    'ACDFGHIL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3F', 82: '3A', 85: '3G', 87: '3L'},
    'ACDFGHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3D'},
    'ACDFGHJL': {74: '3D', 77: '3F', 79: '3C', 80: '3H', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDFGHKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3F', 82: '3A', 85: '3G', 87: '3L'},
    'ACDFGIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ACDFGIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDFGIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ACDFGJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDFHIJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3F', 82: '3A', 85: '3J', 87: '3I'},
    'ACDFHIJL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3F', 82: '3A', 85: '3J', 87: '3L'},
    'ACDFHIKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3F', 87: '3L'},
    'ACDFHJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3F', 82: '3A', 85: '3J', 87: '3L'},
    'ACDFIJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACDGHIJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ACDGHIJL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDGHIKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ACDGHJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDGIJKL': {74: '3C', 77: '3D', 79: '3I', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACDHIJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACEFGHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ACEFGHIK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3I'},
    'ACEFGHIL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ACEFGHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ACEFGHJL': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACEFGHKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ACEFGIJK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ACEFGIJL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACEFGIKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ACEFGJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACEFHIJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3I'},
    'ACEFHIJL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ACEFHIKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3E', 87: '3L'},
    'ACEFHJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ACEFIJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACEGHIJK': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ACEGHIJL': {74: '3C', 77: '3H', 79: '3E', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACEGHIKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ACEGHJKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACEGIJKL': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACEHIJKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACFGHIJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ACFGHIJL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACFGHIKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ACFGHJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACFGIJKL': {74: '3C', 77: '3F', 79: '3I', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ACFHIJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ACGHIJKL': {74: '3C', 77: '3G', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ADEFGHIJ': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ADEFGHIK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3I'},
    'ADEFGHIL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ADEFGHJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3E'},
    'ADEFGHJL': {74: '3D', 77: '3F', 79: '3H', 80: '3E', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADEFGHKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3G', 87: '3L'},
    'ADEFGIJK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ADEFGIJL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADEFGIKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ADEFGJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADEFHIJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3I'},
    'ADEFHIJL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ADEFHIKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3E', 87: '3L'},
    'ADEFHJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3E', 82: '3A', 85: '3J', 87: '3L'},
    'ADEFIJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ADEGHIJK': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ADEGHIJL': {74: '3D', 77: '3H', 79: '3E', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADEGHIKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ADEGHJKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADEGIJKL': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ADEHIJKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ADFGHIJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'ADFGHIJL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADFGHIKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'ADFGHJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADFGIJKL': {74: '3D', 77: '3F', 79: '3I', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'ADFHIJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'ADGHIJKL': {74: '3D', 77: '3G', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'AEFGHIJK': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3I'},
    'AEFGHIJL': {74: '3F', 77: '3H', 79: '3E', 80: '3I', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'AEFGHIKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3G', 87: '3L'},
    'AEFGHJKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3J', 82: '3A', 85: '3G', 87: '3L'},
    'AEFGIJKL': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'AEFHIJKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'AEGHIJKL': {74: '3A', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'AFGHIJKL': {74: '3F', 77: '3G', 79: '3H', 80: '3K', 81: '3I', 82: '3A', 85: '3J', 87: '3L'},
    'BCDEFGHI': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3H', 85: '3G', 87: '3E'},
    'BCDEFGHJ': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3J', 85: '3G', 87: '3D'},
    'BCDEFGHK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3E'},
    'BCDEFGHL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCDEFGIJ': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BCDEFGIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3E', 85: '3G', 87: '3I'},
    'BCDEFGIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3E', 85: '3G', 87: '3L'},
    'BCDEFGJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BCDEFGJL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDEFGKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3E', 85: '3G', 87: '3L'},
    'BCDEFHIJ': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3E'},
    'BCDEFHIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3E', 87: '3I'},
    'BCDEFHIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3H', 85: '3E', 87: '3L'},
    'BCDEFHJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3E'},
    'BCDEFHJL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCDEFHKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3E', 87: '3L'},
    'BCDEFIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3E', 85: '3J', 87: '3I'},
    'BCDEFIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3E', 85: '3J', 87: '3L'},
    'BCDEFIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3I', 85: '3E', 87: '3L'},
    'BCDEFJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3E', 85: '3J', 87: '3L'},
    'BCDEGHIJ': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BCDEGHIK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3I'},
    'BCDEGHIL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCDEGHJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BCDEGHJL': {74: '3C', 77: '3D', 79: '3H', 80: '3E', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDEGHKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCDEGIJK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BCDEGIJL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDEGIKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BCDEGJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDEHIJK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BCDEHIJL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCDEHIKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3I', 87: '3L'},
    'BCDEHJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCDEIJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCDFGHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3D'},
    'BCDFGHIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3I'},
    'BCDFGHIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCDFGHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3D'},
    'BCDFGHJL': {74: '3D', 77: '3F', 79: '3C', 80: '3J', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCDFGHKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCDFGIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BCDFGIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDFGIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BCDFGJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDFHIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BCDFHIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCDFHIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3I', 87: '3L'},
    'BCDFHJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCDFIJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCDGHIJK': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BCDGHIJL': {74: '3C', 77: '3D', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDGHIKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BCDGHJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDGIJKL': {74: '3C', 77: '3D', 79: '3I', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCDHIJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCEFGHIJ': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BCEFGHIK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3I'},
    'BCEFGHIL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCEFGHJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BCEFGHJL': {74: '3C', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCEFGHKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BCEFGIJK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BCEFGIJL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCEFGIKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BCEFGJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCEFHIJK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BCEFHIJL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCEFHIKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3I', 87: '3L'},
    'BCEFHJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCEFIJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCEGHIJK': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BCEGHIJL': {74: '3C', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCEGHIKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BCEGHJKL': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BCEGIJKL': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCEHIJKL': {74: '3C', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCFGHIJK': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BCFGHIJL': {74: '3C', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCFGHIKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BCFGHJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCFGIJKL': {74: '3C', 77: '3F', 79: '3I', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BCFHIJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BCGHIJKL': {74: '3C', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BDEFGHIJ': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BDEFGHIK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3I'},
    'BDEFGHIL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BDEFGHJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3E'},
    'BDEFGHJL': {74: '3D', 77: '3F', 79: '3H', 80: '3E', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BDEFGHKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3G', 87: '3L'},
    'BDEFGIJK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BDEFGIJL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BDEFGIKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BDEFGJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BDEFHIJK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BDEFHIJL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BDEFHIKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3I', 87: '3L'},
    'BDEFHJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BDEFIJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BDEGHIJK': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BDEGHIJL': {74: '3D', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BDEGHIKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BDEGHJKL': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BDEGIJKL': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BDEHIJKL': {74: '3D', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BDFGHIJK': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3I'},
    'BDFGHIJL': {74: '3D', 77: '3F', 79: '3H', 80: '3I', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BDFGHIKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BDFGHJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BDFGIJKL': {74: '3D', 77: '3F', 79: '3I', 80: '3K', 81: '3B', 82: '3J', 85: '3G', 87: '3L'},
    'BDFHIJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BDGHIJKL': {74: '3D', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BEFGHIJK': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3I'},
    'BEFGHIJL': {74: '3F', 77: '3G', 79: '3E', 80: '3I', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BEFGHIKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3G', 87: '3L'},
    'BEFGHJKL': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3H', 85: '3J', 87: '3L'},
    'BEFGIJKL': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BEFHIJKL': {74: '3F', 77: '3H', 79: '3E', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'BEGHIJKL': {74: '3B', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'BFGHIJKL': {74: '3F', 77: '3G', 79: '3H', 80: '3K', 81: '3B', 82: '3I', 85: '3J', 87: '3L'},
    'CDEFGHIJ': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3J', 82: '3H', 85: '3G', 87: '3E'},
    'CDEFGHIK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3H', 85: '3G', 87: '3I'},
    'CDEFGHIL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3E', 82: '3H', 85: '3G', 87: '3L'},
    'CDEFGHJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3E'},
    'CDEFGHJL': {74: '3D', 77: '3F', 79: '3C', 80: '3E', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CDEFGHKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3H', 85: '3G', 87: '3L'},
    'CDEFGIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3J', 85: '3G', 87: '3I'},
    'CDEFGIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3E', 82: '3J', 85: '3G', 87: '3L'},
    'CDEFGIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3I', 85: '3G', 87: '3L'},
    'CDEFGJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3J', 85: '3G', 87: '3L'},
    'CDEFHIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3H', 85: '3J', 87: '3I'},
    'CDEFHIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3E', 82: '3H', 85: '3J', 87: '3L'},
    'CDEFHIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3H', 85: '3E', 87: '3L'},
    'CDEFHJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3H', 85: '3J', 87: '3L'},
    'CDEFIJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3E', 82: '3I', 85: '3J', 87: '3L'},
    'CDEGHIJK': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3I'},
    'CDEGHIJL': {74: '3C', 77: '3D', 79: '3E', 80: '3I', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CDEGHIKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3G', 87: '3L'},
    'CDEGHJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CDEGIJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'CDEHIJKL': {74: '3C', 77: '3D', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'CDFGHIJK': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3I'},
    'CDFGHIJL': {74: '3D', 77: '3F', 79: '3C', 80: '3I', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CDFGHIKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3H', 85: '3G', 87: '3L'},
    'CDFGHJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CDFGIJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'CDFHIJKL': {74: '3D', 77: '3F', 79: '3C', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'CDGHIJKL': {74: '3C', 77: '3D', 79: '3H', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'CEFGHIJK': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3I'},
    'CEFGHIJL': {74: '3C', 77: '3F', 79: '3E', 80: '3I', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CEFGHIKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3G', 87: '3L'},
    'CEFGHJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'CEFGIJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'CEFHIJKL': {74: '3C', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'CEGHIJKL': {74: '3C', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'CFGHIJKL': {74: '3C', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'DEFGHIJK': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3I'},
    'DEFGHIJL': {74: '3D', 77: '3F', 79: '3E', 80: '3I', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'DEFGHIKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3G', 87: '3L'},
    'DEFGHJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3J', 82: '3H', 85: '3G', 87: '3L'},
    'DEFGIJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'DEFHIJKL': {74: '3D', 77: '3F', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'DEGHIJKL': {74: '3D', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'},
    'DFGHIJKL': {74: '3D', 77: '3F', 79: '3H', 80: '3K', 81: '3I', 82: '3J', 85: '3G', 87: '3L'},
    'EFGHIJKL': {74: '3F', 77: '3G', 79: '3E', 80: '3K', 81: '3I', 82: '3H', 85: '3J', 87: '3L'}
}


## Load odds

In [ ]:
with ODDS_PATH.open("r", encoding="utf-8") as file:
    odds_payload = json.load(file)

def pair_key(team_a, team_b):
    return tuple(sorted((team_a, team_b)))


ODDS = {}
for match in odds_payload["matches"]:
    team1 = match["team1_name"]
    team2 = match["team2_name"]
    ODDS[pair_key(team1, team2)] = match

all_group_teams = sorted({team for teams in GROUPS.values() for team in teams})
all_odds_teams = sorted({team for match in odds_payload["matches"] for team in [match["team1_name"], match["team2_name"]]})

missing_from_odds = sorted(set(all_group_teams) - set(all_odds_teams))
if missing_from_odds:
    raise ValueError(f"Group teams missing from odds JSON: {missing_from_odds}")

len(all_group_teams), len(ODDS)

## Simulation functions

In [ ]:
def odds_for(team_a, team_b):
    try:
        return ODDS[pair_key(team_a, team_b)]
    except KeyError as exc:
        raise KeyError(f"No odds found for {team_a} vs {team_b}") from exc


def result_probabilities(team_a, team_b):
    match = odds_for(team_a, team_b)
    if team_a == match["team1_name"]:
        return [match["winteam1_prob"], match["tie_prob"], match["winteam2_prob"]]
    return [match["winteam2_prob"], match["tie_prob"], match["winteam1_prob"]]


def sample_group_score(team_a, team_b, rng):
    match = odds_for(team_a, team_b)
    result = rng.choices(["a", "draw", "b"], weights=result_probabilities(team_a, team_b), k=1)[0]
    scores = match.get("top_scores", [])
    if not scores:
        return sample_result_score(team_a, team_b, rng, result)

    same_order = team_a == match["team1_name"]
    compatible_scores = []
    compatible_weights = []
    for score in scores:
        goals_a = int(score["team1_goals"] if same_order else score["team2_goals"])
        goals_b = int(score["team2_goals"] if same_order else score["team1_goals"])
        if result == "a" and goals_a <= goals_b:
            continue
        if result == "b" and goals_b <= goals_a:
            continue
        if result == "draw" and goals_a != goals_b:
            continue
        compatible_scores.append((goals_a, goals_b))
        compatible_weights.append(max(float(score["probability"]), 0.0))

    if compatible_scores:
        return rng.choices(compatible_scores, weights=compatible_weights, k=1)[0]
    return sample_result_score(team_a, team_b, rng, result)


def sample_result_score(team_a, team_b, rng, result=None):
    if result is None:
        result = rng.choices(["a", "draw", "b"], weights=result_probabilities(team_a, team_b), k=1)[0]
    if result == "a":
        return 1, 0
    if result == "b":
        return 0, 1
    return 1, 1


def knockout_win_probability(team_a, team_b):
    match = odds_for(team_a, team_b)
    if team_a == match["team1_name"]:
        win_a = float(match["winteam1_prob"])
        win_b = float(match["winteam2_prob"])
    else:
        win_a = float(match["winteam2_prob"])
        win_b = float(match["winteam1_prob"])
    tie = float(match["tie_prob"])
    total = win_a + win_b + tie
    raw_probability = 0.5 if total == 0 else (win_a + 0.5 * tie) / total
    return min(max(raw_probability, 1 - KNOCKOUT_MAX_ADVANCE_PROB), KNOCKOUT_MAX_ADVANCE_PROB)


def sample_knockout_winner(team_a, team_b, rng):
    return team_a if rng.random() < knockout_win_probability(team_a, team_b) else team_b


def blank_table(teams):
    return {team: {"team": team, "played": 0, "pts": 0, "gf": 0, "ga": 0, "gd": 0} for team in teams}


def rank_rows(rows):
    return sorted(
        rows,
        key=lambda row: (row["pts"], row["gd"], row["gf"], RATINGS.get(row["team"], 0), row["team"]),
        reverse=True,
    )


def simulate_group(group, teams, rng):
    table = blank_table(teams)
    for i, j in GROUP_MATCH_ORDER:
        team_a = teams[i]
        team_b = teams[j]
        goals_a, goals_b = sample_group_score(team_a, team_b, rng)

        table[team_a]["played"] += 1
        table[team_b]["played"] += 1
        table[team_a]["gf"] += goals_a
        table[team_a]["ga"] += goals_b
        table[team_b]["gf"] += goals_b
        table[team_b]["ga"] += goals_a

        if goals_a > goals_b:
            table[team_a]["pts"] += 3
        elif goals_b > goals_a:
            table[team_b]["pts"] += 3
        else:
            table[team_a]["pts"] += 1
            table[team_b]["pts"] += 1

    for row in table.values():
        row["gd"] = row["gf"] - row["ga"]
        row["group"] = group

    ranked = rank_rows(table.values())
    for index, row in enumerate(ranked, start=1):
        row["group_rank"] = index
    return ranked


def choose_third_place_assignment(qualified_third_groups):
    if len(qualified_third_groups) != 8:
        raise ValueError(f"Expected exactly 8 third-place groups, got {len(qualified_third_groups)}: {qualified_third_groups}")

    key = "".join(sorted(qualified_third_groups))
    try:
        return THIRD_PLACE_ASSIGNMENT_TABLE[key]
    except KeyError as exc:
        raise ValueError(f"No FIFA third-place assignment found for qualified groups key: {key}") from exc


def resolve_slot(slot, group_positions, third_assignment, match_number=None):
    rank = int(slot[0])
    if rank in (1, 2):
        return group_positions[slot[1]][rank]
    if match_number is None:
        raise ValueError(f"Match number is required to resolve third-place slot {slot}")
    assigned_slot = third_assignment.get(match_number)
    if not assigned_slot:
        raise ValueError(f"No third-place assignment found for match {match_number} and slot {slot}")
    assigned_group = assigned_slot[1]
    return group_positions[assigned_group][3]


def simulate_tournament(rng):
    group_rankings = {group: simulate_group(group, teams, rng) for group, teams in GROUPS.items()}
    group_positions = {
        group: {row["group_rank"]: row["team"] for row in rows}
        for group, rows in group_rankings.items()
    }

    thirds = rank_rows([rows[2] for rows in group_rankings.values()])[:8]
    third_assignment = choose_third_place_assignment([row["group"] for row in thirds])

    qualifiers = {group_positions[group][rank] for group in GROUPS for rank in (1, 2)}
    qualifiers.update(row["team"] for row in thirds)

    stage_teams = {"R32": set(qualifiers), "R16": set(), "QF": set(), "SF": set(), "finals": set(), "win": set()}
    winners_by_match = {}

    for slot in R32_TEMPLATE:
        team_a = resolve_slot(slot["a"], group_positions, third_assignment, slot["match"])
        team_b = resolve_slot(slot["b"], group_positions, third_assignment, slot["match"])
        winner = sample_knockout_winner(team_a, team_b, rng)
        winners_by_match[slot["match"]] = winner
        stage_teams["R16"].add(winner)

    stage_map = {"R16": "QF", "QF": "SF", "SF": "finals", "F": "win"}
    for round_key in ["R16", "QF", "SF", "F"]:
        for match_number, left_match, right_match in NEXT_ROUNDS[round_key]:
            team_a = winners_by_match[left_match]
            team_b = winners_by_match[right_match]
            winner = sample_knockout_winner(team_a, team_b, rng)
            winners_by_match[match_number] = winner
            stage_teams[stage_map[round_key]].add(winner)

    return stage_teams


## Run iterations and write JSON

In [ ]:
rng = random.Random(SEED)
stages = ["R32", "R16", "QF", "SF", "finals", "win"]
counts = {team: {stage: 0 for stage in stages} for team in all_group_teams}

for iteration in range(N_ITERATIONS):
    stage_teams = simulate_tournament(rng)
    for stage in stages:
        for team in stage_teams[stage]:
            counts[team][stage] += 1

results = [
    {"team": team, **counts[team]}
    for team in sorted(counts)
]

payload = {
    "metadata": {
        "tournament": "WC-2026",
        "iterations": N_ITERATIONS,
        "seed": SEED,
        "odds_source": str(ODDS_PATH.relative_to(PROJECT_ROOT)),
        "format": "12 groups of 4; group winners, runners-up, and 8 best third-place teams advance to R32; FIFA 2026 knockout slot constraints",
        "stage_counts_are_entries": True,
        "knockout_tie_policy": "Ties are split 50/50 between teams to approximate penalty shootouts.",
        "knockout_max_advance_prob": KNOCKOUT_MAX_ADVANCE_PROB,
        "knockout_probability_policy": "Knockout advancement probabilities split ties 50/50 for penalties and cap either team at 80% to advance.",
    },
    "teams": results,
}

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    json.dump(payload, file, ensure_ascii=False, indent=2)

OUTPUT_PATH, pd.DataFrame(results).sort_values("win", ascending=False).head(12)
